# Import Libraries

In [ ]:
import os
import pickle
import random
import itertools

# Analisi dati e Visualizzazione
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: Preprocessing e Dimensionality Reduction
from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    LabelEncoder,
    OneHotEncoder
)


# Scikit-learn: Model Selection e Pipeline
from sklearn.model_selection import (
    train_test_split,
)
from sklearn.pipeline import Pipeline

# Metriche di Valutazione
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    classification_report,
    roc_auc_score,
    roc_curve,
    PrecisionRecallDisplay
)

# Deep Learning (PyTorch) e Monitoraggio
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

# Our packages
from support_modules.utils import *
from support_modules.plot import *


from sklearn.compose import ColumnTransformer


from sklearn.compose import make_column_selector
from sklearn import set_config
set_config(transform_output="pandas")



# Global Variables

In [ ]:
SEED = 42
FILENAME = "../data/train.csv"

# Cerca la GPU
if torch.backends.mps.is_available():
    print("MPS device is available.")
    device = torch.device("mps")
elif torch.cuda.is_available():
    print("CUDA device is available.")
    device = torch.device("cuda")
else:
    print("No GPU acceleration available.")
    device = torch.device("cpu")

In [ ]:
def fix_random(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

fix_random(SEED)

# Load the dataset

In [ ]:
df = pd.read_csv(FILENAME, encoding='ISO-8859-1', sep=",")

rows = df.shape[0]
cols = df.shape[1]
print("# Righe: " + str(rows)+ " # Colonne: "+str(cols) + "\n")

# Preprocessing

## 1. Remove duplicates rows and columns

In [ ]:
# Individua se esistono colonne con lo stesso nome
# Se esistono, allora se le colonne sono duplicati perfetti, droppiamo il duplicato
# Se esistono, ma nono sono perfetti duplicati, per intervenire consciamente sarebbe necessario avere maggior domain knowledge
feature_list = df.columns.to_list()
has_duplicate_cols = len(feature_list) != len(set(feature_list))
print("Ci sono colonne con lo stesso nome?", has_duplicate_cols)

if has_duplicate_cols:
    df2 = df.T.drop_duplicates().T


# Rimuovi righe duplicate
df.drop_duplicates(inplace=True)


##################################################
print("Nuovo # Righe: " + str(rows)+ " Nuovo # Colonne: "+str(cols) + "\n")


## 2. Label extraction and train-test splitting

In [ ]:
X = df.drop(columns=["grade"])
y = df["grade"]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

In [ ]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)

## 3. Pipeline

In [ ]:
from support_modules.preprocessing import *

In [ ]:
from support_modules.constants import *

In [ ]:
remainder_pipeline = Pipeline([
    ('impute_median', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('impute_unknown', SimpleImputer(strategy='constant', fill_value='Unknown'), categorical_to_unknown_cols),
        ('impute_0', SimpleImputer(strategy='constant', fill_value=0), fill_zero_cols),
        ('impute_10k', SimpleImputer(strategy='constant', fill_value=999), fill_big_cols),
        ('impute_mode_cat',SimpleImputer(strategy='most_frequent'), fill_to_mode_cat),
        ('impute_mode_num', SimpleImputer(strategy='most_frequent'), fill_to_mode_num),
    ],
    remainder=remainder_pipeline,
    verbose_feature_names_out=False
)


encoding = ColumnTransformer(
    transformers=[
        # Applica categorical_pipe a TUTTE le colonne di tipo object o category
        ('cat_step', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), make_column_selector(dtype_include=['object'])),
    ], 
    # Tutto ciò che è numerico e non è stato toccato sopra, viene scalato qui
    remainder="passthrough", 
    verbose_feature_names_out=False
)

# Feed Forward Neural Network

In [ ]:
# Define the Data Layer

class MyDataset(Dataset):
    def __init__(self, X, y):

        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

        self.num_samples = X.shape[0]         # Definisco numero di samples
        self.num_features = X.shape[1]        # Definisco numero di feature
        self.num_classes = len(np.unique(y))  # Definisco numero di classi


    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.X[idx, :], self.y[idx]

In [ ]:
def test_model(model, data_loader, device):
    model.eval()
    y_pred = []
    y_test = []

    for data, targets in data_loader:
        data, targets = data.to(device), targets.to(device)
        y_pred += model(data)
        y_test += targets

    y_test = torch.stack(y_test).squeeze()
    y_pred = torch.stack(y_pred).squeeze()
    y_pred_c = y_pred.argmax(dim=1, keepdim=True).squeeze()

    return y_test, y_pred_c, y_pred

In [ ]:
def train_model(model, criterion, optimizer, num_epochs, scheduler, 
                train_loader, val_loader, device, writer, log_name="best_model"):
    
    train_losses = []
    val_losses = []
    
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        model.train()
        running_train_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_train_loss += loss.item() * inputs.size(0)
        
        epoch_train_loss = running_train_loss / len(train_loader.dataset)
        train_losses.append(epoch_train_loss)
        
        model.eval()
        running_val_loss = 0.0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                running_val_loss += loss.item() * inputs.size(0)
        
        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        val_losses.append(epoch_val_loss)
        
        writer.add_scalar('Loss/train', epoch_train_loss, epoch)
        writer.add_scalar('Loss/val', epoch_val_loss, epoch)
        
        print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}')
        
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), f"models/{log_name}")
        
        scheduler.step()
    
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, num_epochs + 1), train_losses, label='Training Loss', marker='o')
    plt.plot(range(1, num_epochs + 1), val_losses, label='Validation Loss', marker='s')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss over Epochs')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    
    plt.savefig(f'models/{log_name}_loss_plot.png', dpi=300)
    plt.close()
    
    return model

In [ ]:
preprocessing_pipeline = Pipeline([
    ('drop_leakage', ColumnDropper(columns=loan_performance_data_leakage + settlement_data_leakage + hardship_data_leakage + other_leakage + loan_contract_interest_rate)),
    ('drop_non_significant', ColumnDropper(columns=other_non_significant)),
    ('drop_high_nan', HighNanDropper(threshold=0.9)),
    ('drop_joint_and_secondary', ColumnDropper(columns=joint_and_secondary_cols)),
    ('high_correlation', HighlyCorrelatedDropper(threshold=0.95)),
    ('feature_extraction', NumericExtractor(columns=number_from_string_cols)),
    ('fico_average', FeatureAverager(columns=average_cols)),
    ('date_diff', DateDifferenceTransformer(reference_col=date_diff_reference, target_cols=date_diff_target)),
    ('rounding_int', RoundToIntTransformer(columns=round_to_nearest_int)),

    ('preprocessor', preprocessor),
    ('encoding', encoding),

    ('scaler', MinMaxScaler())
])

In [ ]:
X_train = preprocessing_pipeline.fit_transform(X_train).to_numpy()
X_val = preprocessing_pipeline.transform(X_val).to_numpy()

# Salva il preprocessor
if not os.path.exists('models'):
    os.makedirs('models')
pickle.dump(preprocessing_pipeline, open("models/ff_preprocessor.save", 'wb'))
print("Preprocessor salvato in models/ff_preprocessor.save")

# Create the torch dataset
train_subset = MyDataset(X_train,y_train)
val_subset = MyDataset(X_val,y_val)

In [ ]:
# Let's define a new architecture inlcuding also Dropout
class FeedForward_NN(nn.Module):
    def __init__(self, input_size, num_classes, hidden_size, dropout_rate, depth=1):
        super(FeedForward_NN, self).__init__()

        model = [
            nn.Linear(input_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        ]

        block = [
            nn.Linear(hidden_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        ]

        for i in range(depth):
            model += block

        self.model = nn.Sequential(*model)

        self.output = nn.Linear(hidden_size, num_classes)


    def forward(self, x):
        h = self.model(x)
        out = self.output(h)
        return out

In [ ]:

# --- 1. Definizione della Griglia degli Iperparametri ---
param_grid = {
    'batch_size': [256, 512, 1024],
    'hidden_size': [100, 150, 200],
    'depth': [1, 2],
    'dropout_rate': [0.2, 0.3],
    'learning_rate': [0.001, 0.01]
}

# Parametri fissi
num_epochs = 50
gamma = 0.5
step_size = 10

# Genera tutte le combinazioni possibili
keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

best_val_acc_overall = 0.0
best_config = None
best_run_name = "" # Inizializza fuori dal ciclo for

# --- 2. Ciclo di Iterazione ---
for i, config in enumerate(combinations):
    # Estrazione parametri correnti
    bs = config['batch_size']
    hs = config['hidden_size']
    d = config['depth']
    dr = config['dropout_rate']
    learning_rate = config['learning_rate']

    print(f"\n[Run {i+1}/{len(combinations)}] Testing: {config}")

    # Fix the seed for reproducibility ad ogni inizio ciclo
    fix_random(SEED)

    # Start tensorboard con nome esperimento dinamico
    exp_name = f"runs/FFNN_bs{bs}_d{d}_h{hs}_dr{dr}_lr{learning_rate}"
    writer = SummaryWriter(log_dir=exp_name)

    # Create relative dataloaders
    train_loader = DataLoader(train_subset, batch_size=bs, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=bs)

    # Define the architecture, loss and optimizer
    # Assicurati che l'ordine degli argomenti in FeedForward_NN sia corretto
    model = FeedForward_NN(train_subset.num_features, train_subset.num_classes, hs, dr, d)
    model.to(device)

    # Add model graph to TensorBoard (solo alla prima iterazione per non appesantire)
    if i == 0:
        data_sample, _ = next(iter(train_loader))
        writer.add_graph(model, data_sample.to(device))

    criterion = torch.nn.CrossEntropyLoss()
    #optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

    # Nome univoco per il miglior modello di QUESTA specifica run
    current_run_model_name = f"best_model_run_{i}"

    # Train the model
    # Nota: la tua funzione train_model salva già internamente il miglior modello della run in 'models/'
    model = train_model(model, criterion, optimizer, num_epochs, scheduler,
                        train_loader, val_loader, device, writer, log_name=current_run_model_name)

    # --- 3. Validazione Finale e Confronto ---
    model.load_state_dict(torch.load(f"models/{current_run_model_name}", weights_only=True))
    model.to(device)

    # Recupero predizioni
    labels_val, preds_val, _ = test_model(model, val_loader, device)

    # CONVERSIONE IN NUMPY per sklearn
    y_true = labels_val.cpu().numpy()
    y_pred = preds_val.cpu().numpy()

    # CALCOLO METRICHE
    acc = (preds_val == labels_val).float().mean().item()
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    f1_weighted = f1_score(y_true, y_pred, average='weighted') # Media pesata tra le classi

    print(f"--- Risultati Run {i+1} ---")
    print(f"Accuracy Standard: {acc:.4f}")
    print(f"Balanced Accuracy: {balanced_acc:.4f}")
    print(f"F1 Score (weighted):  {f1_weighted:.4f}")

    # Stampa il report completo (Precision, Recall per ogni classe)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    # LOG SU TENSORBOARD (Hparams)
    # Aggiungiamo tutte le metriche per poterle confrontare nella tabella Hparams di TensorBoard
    writer.add_hparams(
        config,
        {
            "hparam/accuracy": acc,
            "hparam/balanced_accuracy": balanced_acc,
            "hparam/f1_weighted": f1_weighted
        }
    )


    # SALVATAGGIO BEST MODEL (Consiglio: usa la Balanced Accuracy o F1 se i dati sono sbilanciati)
    if balanced_acc > best_val_acc_overall:
        best_val_acc_overall = balanced_acc # Ora stiamo ottimizzando per Balanced Accuracy
        best_config = config
        best_run_name = exp_name
        torch.save(model.state_dict(), "models/ff.pth")
        
        # Salvo i metadati del modello
        ff_metadata = {
        'input_size': train_subset.num_features,
        'num_classes': train_subset.num_classes,
        'hidden_size': hs,
        'dropout_rate': dr,
        'depth': d
        }

        pickle.dump(ff_metadata, open("models/ff_metadata.save", 'wb'))
        print(f"!!! Nuovo miglior modello trovato nella run: {best_run_name} !!!")

    # Close tensorboard writer
    writer.flush()
    writer.close()

print("\n" + "="*40)
print(f"GRID SEARCH TERMINATA")
print(f"Miglior accuratezza: {best_val_acc_overall:.4f}")
print(f"Miglior configurazione: {best_config}")

In [ ]:
import json

# 1. Creiamo un report completo
best_model_summary = {
    "model_name": "FeedForward_NN",
    "best_config": best_config,
    "metrics": {
        "best_balanced_accuracy": best_val_acc_overall,
    },
    "tensorboard_log_dir": best_run_name, # Qui avrai il percorso esatto della run
    "architecture_preview": str(model)     # Salva la lista dei layer e parametri
}

# 2. Salvataggio del file JSON
with open("models/ff_best.json", "w") as f:
    json.dump(best_model_summary, f, indent=4)

print("\n" + "="*40)
print("✅ SALVATAGGIO COMPLETATO")
print(f"Pesi salvati in: models/BEST_OVERALL_MODEL.pth")
print(f"Info e Run salvate in: models/BEST_MODEL_INFO.json")
print(f"Per vedere i grafici, cerca su TensorBoard: {best_run_name}")
print("="*40)